# Project 1 ANSC 4040 Data Split

This is a notebook for Project 1 that will be used for making the test, train and validation data split. The plan outlined below organizes the split chronologically by date and by cow. With the use of the tree-based model I plan to use, it it vital that all test cows are prepresented in the training at least once in order to improve the accuract of the model. 

In [15]:
#import pandas as pd
import pandas as pd

In [16]:
# Add file path and load the dataset
path = 'clean_data.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,NaN,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,NaN,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


# Plan for the split

1. Exlude missing values into a new data set- "missing_data"
2. Sort by date and by cow
3. Split by time order, ensuring each cow present in the test set is represented in the training set
     Keep the most recent dates in the test set (10%)
     Use earlier dates for training (70%)
     Use a validation window between them (20%)

# Exlude missing values into a new data set - "missing_data"

In [17]:
# Remove rows with any missing values from the clean dataset
before_rows = len(data)

data = data.dropna().copy()

after_rows = len(data)

print(f"Rows before removing missing values: {before_rows}")
print(f"Rows after removing missing values: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

# Keep rows with missing values in a separate dataset
missing_data = data[data.isna().any(axis=1)].copy()

# Keep only rows without missing values
clean_data = data.dropna().copy()

missing_data.to_csv("missing_data.csv", index=False)
clean_data.to_csv("clean_data_no_missing.csv", index=False)


Rows before removing missing values: 8285371
Rows after removing missing values: 6599539
Rows removed: 1685832


# Sort Remaining Data by Date and by COW
Keep the most recent dates in the test set (10%)
Use earlier dates for training (70%)
Use a validation window between them (20%)

ENSURING EACH COW FROM TEST SPLIT IS REPRESENTED IN TRAINING

In [18]:
# Add file path and load the dataset without any mising values
path = 'clean_data_no_missing.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
3,5.803650e+18,1.0,394.0,Open,2019-12-14,3.492661,4.399846,8.173735,11.793402,198,1
4,-1.713215e+18,5.0,288.0,Pregnant,2020-05-11,3.311224,4.300056,7.466130,8.572896,153,2


In [19]:
# Identify the date column
date_columns = [col for col in data.columns if "date" in col.lower()]
if not date_columns:
    raise ValueError("No date-like column found in the dataset.")

date_col = date_columns[0]

# Make sure the date column is datetime
data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
data = data.dropna(subset=[date_col]).copy()

# Make sure the cow ID column exists
cow_col = "AnimalId"
if cow_col not in data.columns:
    raise ValueError(f"Column '{cow_col}' was not found. Update cow_col to your actual cow ID column.")

# Sort by date and cow
data = data.sort_values([date_col, cow_col]).reset_index(drop=True)

# Determine the last date for each cow
last_date_by_cow = data.groupby(cow_col)[date_col].max().sort_values()

# Create proportional splits by cow
n_cows = len(last_date_by_cow)
test_n = max(1, int(round(n_cows * 0.10)))
val_n = max(1, int(round(n_cows * 0.20)))
train_n = max(1, n_cows - test_n - val_n)

# Most recent cows go to test, next recent to validation, rest to training
test_cows = last_date_by_cow.iloc[-test_n:].index.tolist()
val_cows = last_date_by_cow.iloc[-(test_n + val_n):-test_n].index.tolist()
train_cows = last_date_by_cow.iloc[:train_n].index.tolist()

# Keep the latest record(s) of each test cow in the test set
data = data.merge(
    last_date_by_cow.rename("last_date"),
    left_on=cow_col,
    right_index=True
)

test = data[
    data[cow_col].isin(test_cows) & (data[date_col] == data["last_date"])
].copy()

# Training set includes all earlier records for test cows plus all train cows
train = data[
    (data[cow_col].isin(train_cows)) |
    (
        data[cow_col].isin(test_cows) &
        (data[date_col] < data["last_date"])
    )
].copy()

# Validation set uses the val cows
val = data[
    data[cow_col].isin(val_cows)
].copy()

# Remove helper column
train = train.drop(columns=["last_date"])
val = val.drop(columns=["last_date"])
test = test.drop(columns=["last_date"])

print(f"Total rows: {len(data)}")
print(f"Train rows: {len(train)}")
print(f"Validation rows: {len(val)}")
print(f"Test rows: {len(test)}")

train.head()

Total rows: 6599539
Train rows: 5059709
Validation rows: 1537643
Test rows: 2187


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-9.208799e+18,3.0,219.0,Pregnant,2019-06-14,4.082331,4.799007,9.448329,14.424237,211,3
1,-9.208799e+18,3.0,219.0,Pregnant,2019-06-14,4.218409,4.799007,9.126278,15.785014,225,1
2,-9.208799e+18,3.0,219.0,Pregnant,2019-06-14,3.991613,4.898798,9.099063,15.013907,220,2
3,-9.203759e+18,3.0,83.0,Bred,2019-06-14,3.084428,3.302152,6.273182,14.741752,282,2
4,-9.203759e+18,3.0,83.0,Bred,2019-06-14,2.993710,3.401943,6.450084,15.694296,307,3


In [20]:
# Save the train, validation, and test splits as separate CSV files
train.to_csv("train_data.csv", index=False)
val.to_csv("val_data.csv", index=False)
test.to_csv("test_data.csv", index=False)

print("Saved: train_data.csv, val_data.csv, test_data.csv")

Saved: train_data.csv, val_data.csv, test_data.csv
